<a href="https://colab.research.google.com/github/thedatasense/robust-med-mllm-experiments/blob/main/notebooks/08_Tiny_VLM_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Compact Vision-Language Model

Inspired by Nate Nethercott’s [vision-language-from-scratch tutorial](https://medium.com/@natenethercott/vision-language-models-from-scratch-in-colab-cd073a753b8a).

## Preparing the Environment

To train our multimodal system in Google Colab, we'll begin by installing required packages and importing essential libraries.

In [ ]:
!pip install -q torch transformers datasets accelerate bitsandbytes

In [ ]:
# We'll have ~16GiB of vram to work with for our training which is more than enough
!nvidia-smi

Sun Sep 15 10:56:03 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   59C    P8              10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## Architecture Overview

We'll use [CLIP's ResNet-50](https://huggingface.co/openai/clip-vit-base-patch32) as the image encoder, paired with a lightweight GPT-2 transformer as the textual decoder.

## Constructing the Model

Our model comprises three main parts:

1. **Vision Encoder**: Processes images into feature embeddings.
2. **Language Decoder**: Generates text based on projected embeddings.
3. **Projection Layer**: Aligns the encoder's outputs with the decoder’s vocabulary space.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, SiglipImageProcessor, SiglipVisionModel

Below is the interface our model class must adhere to:

In [ ]:
from abc import ABC
from typing import Optional

class VLMMetaArchitecture(ABC):
  def prepare_inputs(self, input_ids=None, attention_mask=None, inputs_embeds=None, labels=None, images=None):
      """
      Prepare the inputs for the model.
      """
      raise NotImplementedError

  def forward(self, input_ids, attention_mask, position_ids, past_key_values, inputs_embeds, labels, use_cache, output_attentions, output_hidden_states, images, return_dict):
      """
      Forward pass of the model. Wrapper around huggingface AutoModelForCausalLM forward method.
      """
      raise NotImplementedError

  @torch.no_grad()
  def generate(self, input_ids, images, **kwargs):
      """
      Generate text from the model. Wrapper around huggingface AutoModelForCausalLM generate method.
      """
      raise NotImplementedError


## a comment on prepare_inputs

LLMs work by turning text->tokens->embeddings and performing matrix operations on the embedding vectors. Vision transformers (ViT) convert images into patch embeddings;

<img src="https://production-media.paperswithcode.com/methods/Screen_Shot_2021-01-26_at_9.43.31_PM_uI4jjMq.png" alt="isolated" width="600"/>

By projecting the vision embeddings to the same dimension as the token embeddings we can "stick" the image vectors beside the text ones and give those vectors to the llm.

The pseudo-code goes something like this:
```
# get text vectors
tokens = tokenizer(text)
text_embeds = llm.embedding(tokens)

# get image patch vectors
img_embeds = vision(images)
img_embeds = nn.Linear(vision_dim, llm_dim)(img_embeds) #R^n->R^m

# concatenate
all_embeds = torch.cat([img_embeds, text_embeds], dim=1)

# model forward/generate
loss, logits = llm.forward(inputs_embeds = all_embeds)
```

The sequences we'll give to our llm will follow a basic prompt template that looks like:
```
<bos><img><img_tokens></img>user:\nDescribe this image.<eos>assistant:\nThis image shows ...<eos>
```

When training LLMs we also need to ignore loss over our prompt tokens, which means in our labels we'll need to manually zero out all the tokens corresponding to image patches or user prompts.

*Okay now for the code ...*

In [ ]:
class VLM(VLMMetaArchitecture, nn.Module):
    def __init__(self, vision, llm, tokenizer):
        super().__init__()
        self.vision = vision
        self.llm = llm
        self.tokenizer = tokenizer
        self.configure_tokenizer()

        # define the connector layer
        self.connector = nn.Sequential(
            nn.Linear(vision.config.hidden_size, llm.config.hidden_size),
            nn.SiLU(),
            nn.Linear(llm.config.hidden_size, llm.config.hidden_size)
        )

        # weight tying uses the embedding layer as the lm_head
        self.llm.tie_weights()


    def freeze(self):
        # freeze all model parameters
        for param in self.parameters():
            param.requires_grad = False

        # unfreeze connector
        for p in self.connector.parameters():
          p.requires_grad = True

        # unfreeze llm embedding weights
        for p in self.llm.embed_tokens.parameters():
          p.requires_grad = True


    def configure_tokenizer(self):
        self.tokenizer.padding_side = 'right'
        self.tokenizer.add_tokens(["<img>", "</img>"])

        # resize token embedding matrix and lm_head
        self.llm.resize_token_embeddings(len(self.tokenizer))

    @property
    def device(self):
        return self.llm.device
    @property
    def img_token(self):
        return self.tokenizer.encode("<img>", add_special_tokens=False)[0]
    @property
    def end_img_token(self):
        return self.tokenizer.encode("</img>", add_special_tokens=False)[0]

    @property
    def embed_tokens(self):
        return self.llm.model.embed_tokens


    def prepare_inputs(
        self,
        input_ids=None,
        attention_mask=None,
        inputs_embeds=None,
        labels=None,
        images=None
    ):
        """
        Prepare the inputs for the model.
        """
        if images is None:
            return input_ids, attention_mask, inputs_embeds, labels #no-op; standard llm preprocessing

        bsz = len(images)

        vision_embeds = self.vision(images) # NOTE: WE STILL NEED TO DEFINE THIS !
        vision_embeds = self.connector(vision_embeds)

        # bos
        bos_token = torch.tensor(self.tokenizer.bos_token_id, device=self.device).unsqueeze(0)
        bos_embeds = self.embed_tokens(bos_token).repeat((bsz,1,1))

        # embed <img> and </img>
        img_token = torch.tensor(self.img_token, device=self.device).unsqueeze(0)
        img_embeds = self.embed_tokens(img_token).repeat((bsz,1,1))
        end_img_token = torch.tensor(self.end_img_token, device=self.device).unsqueeze(0)
        end_img_embeds = self.embed_tokens(end_img_token).repeat((bsz,1,1))

        if input_ids is not None:
            # embeddings
            text_embeds = self.embed_tokens(input_ids)
            input_ids = None

            inputs_embeds = torch.cat((bos_embeds, img_embeds, vision_embeds, end_img_embeds, text_embeds[:,1:,:]), dim=1)


            # attention_mask
            _, vis_len, _ = vision_embeds.shape
            additional_len = 1 + 1 #added <img> and </img>
            attention_mask = torch.cat((torch.ones((bsz, vis_len+additional_len), device=self.device), attention_mask), dim=1)


            # labels
            if labels is not None:
                labels = labels[:,1:]
                additional_len+=1 #added </s>
                labels_prefix = torch.tensor([-100]*(vis_len+additional_len), device = self.device) #tokens with -100 get ignored in loss fn

                labels_prefix = labels_prefix.repeat((bsz, 1))
                labels = torch.cat((labels_prefix, labels), dim=1)


        else:
            inputs_embeds = torch.cat((bos_embeds, img_embeds, vision_embeds, end_img_embeds), dim=1)
            attention_mask = torch.ones(inputs_embeds.shape[:-1], device=self.device)

        return None, attention_mask, inputs_embeds, labels


    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        position_ids=None,
        past_key_values=None,
        inputs_embeds=None,
        labels=None,
        use_cache=None,
        output_attentions=None,
        output_hidden_states=None,
        images=None,
        return_dict=True,
    ):
        """
        Pass the concatenated [img, text] embedding through the llm.
        """
        assert input_ids is not None or images is not None, "You can't forward without text and/or images!"

        input_ids, attention_mask, inputs_embeds, labels = self.prepare_inputs(input_ids, attention_mask, inputs_embeds, labels, images)

        return self.llm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            labels=labels,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict
        )

    @torch.no_grad()
    def generate(
        self,
        input_ids=None,
        images=None,
        **kwargs,
    ):
        if "inputs_embeds" in kwargs:
            raise NotImplementedError("`inputs_embeds` is not supported")

        attention_mask = kwargs.pop("attention_mask", None)

        if images is not None:
              _, attention_mask, inputs_embeds, _ = self.prepare_inputs(input_ids, attention_mask, None, None, images)
        else:
            # if no images passed we just use the text embeddings
            inputs_embeds = self.llm.model.embed_tokens(input_ids)

        return self.llm.generate(
            attention_mask=attention_mask,
            inputs_embeds=inputs_embeds,
            **kwargs
        )

We still need to write a small wrapper around the vision model so this all works. We'll use the same strategy as proposed in the [minigptv2 paper](https://arxiv.org/abs/2310.09478) where $n$ adjacent vision tokens get concatenated together. Doing this lets us save on memory later since memory grows quadratically with respect to input sequence length in the transformer self-attention blocks.

To implement this we'll introduce a hyperparameter $r$ which defines how many consecutive patch embeddings we want to stack.

<img src="https://cdn-images-1.medium.com/max/1600/1*jB0b5-YGe08iTwRJX0VP5Q.png"></img>



In [ ]:
class Vision(nn.Module):
    def __init__(self, vision, processor, r:int = 1):
        super().__init__()
        self.vision = vision
        self.image_processor = processor
        self.r = r #how many adjacent tokens to concatenate

        # reshape sequence
        setattr(self.vision.config, 'hidden_size', self.r*self.vision.config.hidden_size)

    @property
    def config(self):
        return self.vision.config
    @property
    def device(self):
        return self.vision.device

    def vit_forward(self, x):
        x = self.image_processor(x, return_tensors = 'pt')['pixel_values']
        x = self.vision(x.to(self.device), output_hidden_states=True)
        x = x['hidden_states'][-1] # which depth to choose hidden features from
        return x

    def forward(self, x, attention_mask = None):
        # vision-backbone
        x = self.vit_forward(x)
        b,s,_ = x.shape

        # concatenate adjacent tokens a la minigpt4-v2
        return x.reshape((b, s//self.r, -1))


In [ ]:
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# language
llm_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
llm = AutoModelForCausalLM.from_pretrained(llm_id,
                                           #quantization_config=quant_config
                                          )
tokenizer = AutoTokenizer.from_pretrained(llm_id)

# vision
vision_id="google/siglip-so400m-patch14-384"
vision=SiglipVisionModel.from_pretrained(vision_id)
img_processor=SiglipImageProcessor.from_pretrained(vision_id)
vision = Vision(vision, img_processor, r=9)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# mllm
model = VLM(vision, llm, tokenizer).to("cuda")

In [ ]:
model

VLM(
  (vision): Vision(
    (vision): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(729, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (self_attn): SiglipSdpaAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (mlp): SiglipMLP(
                (activation_fn): PytorchGELUTanh()
                (fc

In [ ]:
from PIL import Image
import requests
import io

url = "https://c.files.bbci.co.uk/C8AF/production/_120357315_moshpit_crowdsurfer_gettyimages_976.jpg"
img = Image.open(io.BytesIO(requests.get(url).content))

template = tokenizer.bos_token + "{prompt}" + tokenizer.eos_token
inputs = tokenizer('this is a test', return_tensors = 'pt')
inputs['labels'] = inputs['input_ids'].clone()
inputs['images'] = [img,]

for k, v in inputs.items():
    if isinstance(v, torch.Tensor):
        inputs[k] = v.to(model.device)

outputs = model.forward(**inputs, return_dict=True)
print(outputs['logits'])

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


tensor([[[-4.6822,  0.9866,  4.5126,  ..., -4.2285,  1.3128,  0.1474],
         [-5.3174, -4.8713,  6.4350,  ..., -0.5258, -1.1517, -0.8712],
         [-2.1804, -2.4592,  1.5738,  ...,  1.1067, -0.1072, -1.1964],
         ...,
         [-6.5687, -6.4010,  5.0208,  ..., -1.8882,  0.0972, -1.4747],
         [-7.7164, -7.7324,  6.2165,  ..., -3.5155,  2.7111, -0.6949],
         [-4.9667, -4.6441,  7.3285,  ..., -0.7788,  0.8053, -0.6010]]],
       device='cuda:0', grad_fn=<UnsafeViewBackward0>)


Let's do a quick sanity check to understand the output logits shape;

* images are turned into vector of size $[1,729,1152]$
* we combine $n=9$ adjacent image patches
* our text inputs should be `len(tokenizer("this is a test"))` long
  * we also need the `<s>`, and two `</s>` from the prompt template above
* we add `<img>` and `</img>` around our image tokens

Notice also how much of our input sequence is made up of just image tokens!

In [ ]:
# this works out to
expected_num_tokens = len(tokenizer("this is a test", add_special_tokens=False)) + 2 + 729//9 + 3
print(f"expected num tokens: {expected_num_tokens}")

# actual
_, bzs, _ = outputs['logits'].shape
print(f"num tokens: {bzs}")

# image tokens
print(f"num img tokens: {729//9}")

expected num tokens: 88
num tokens: 88
num img tokens: 81


In [ ]:
# before training our model can't make sense of the img tokens ...
generated = model.generate(**inputs, temperature=0.7, max_new_tokens=64, do_sample=True)
tokenizer.batch_decode(generated)

['nezza…\nwithslt more distantthernecdotes with the same daughteretwithness with same\n00404040040 years on the\nwith the same40000400404040404040404']

![alt-text](https://c.files.bbci.co.uk/C8AF/production/_120357315_moshpit_crowdsurfer_gettyimages_976.jpg)

# training

## trainer class
This is another instance where we'll make use of the huggingface ecosystem. The [trainer class](https://huggingface.co/docs/transformers/en/main_classes/trainer) is an incredibly useful resource we'll make use of here to save us time and improve clarity.

We'll be adding some custom features to the base Trainer object to avoid some common pitfalls but our modifications will be small.

## dataset
To get a baseline MLLM working we need a rich dataset of images and text. For our training we'll use a small subset of [UCSC-VLAA/Recap-DataComp-1B](https://huggingface.co/datasets/nnethercott/Recap-DataComp-100K) consisting of 100K image-text pairs.

Samples are basic captions (without user instructions/prompt) resembling:

><div style="text-align: center;">
<img src="https://images-na.ssl-images-amazon.com/images/I/41kzIaKFE+L._AC._SR180,230.jpg">
><div class="caption">"A modern coffee machine with a digital display and two white coffee cups filled with coffee is shown. The machine has a stainless steel finish and is accompanied by a milk frothing pitcher with a white liquid inside. The coffee machine is placed on a surface with a white background."</div>
</div>

## custom collating function
A collating function (`collate_fn`) is something that gets applied to your batches before passing them to the model. Usually we inject some dynamic behaviour like padding to the max length in the batch.

In our case we would like our collating function to address the following points:
* downloading all images before training is time consuming
* we can use concurrency in the dataloader pre-fetching and a custom collate fn to download images right before we need them
* mitigate our colab session RAM usage


In [ ]:
# load data
from datasets import load_dataset

dataset = load_dataset("nnethercott/Recap-DataComp-100K", split="train[:10000]")

# a row of the dataset
dataset[0]

{'url': 'https://images-na.ssl-images-amazon.com/images/I/41kzIaKFE+L._AC._SR180,230.jpg',
 're_caption': 'A modern coffee machine with a digital display and two white coffee cups filled with coffee is shown. The machine has a stainless steel finish and is accompanied by a milk frothing pitcher with a white liquid inside. The coffee machine is placed on a surface with a white background.',
 'org_caption': 'Saeco Xelsis Automatic Espresso Machine, SM7685/04, Stainless Steel',
 'sha256': '501ac5e3066ff4cdad97dafd8916a65673b5e6d961c6e31e59dbeadd7fd7e3c7',
 'key': '000000000008',
 're_clip_score': 0.0,
 'org_clip_score': 0.0,
 're_length': 50,
 'org_length': 8,
 're_gpt4v_score': 0,
 'org_gpt4v_score': 0}

In [ ]:
tokenizer.padding_side='right'

# preprocessing function
def preprocess(samples):
  """
  tokenizes input ids and labels
  """
  # tokenize samples
  text = samples['re_caption']
  inputs = tokenizer(text, padding='max_length', max_length=128, truncation=True)
  labels = inputs['input_ids']
  inputs['urls'] = samples['url']

  # ignore pad tokens in labels
  labels_t = torch.tensor(labels)
  labels_t.masked_fill_(labels_t == tokenizer.pad_token_id, -100)
  labels = labels_t.tolist()

  inputs['labels'] = labels

  return inputs

dataset = dataset.map(preprocess, batched=True, remove_columns=dataset.column_names)

# notice all the -100's which get ignored during training
print(dataset[0]['labels'])

[1, 319, 5400, 26935, 4933, 411, 263, 13436, 2479, 322, 1023, 4796, 26935, 2723, 567, 10423, 411, 26935, 338, 4318, 29889, 450, 4933, 756, 263, 380, 475, 2222, 22973, 8341, 322, 338, 21302, 491, 263, 27274, 14671, 1918, 15905, 261, 411, 263, 4796, 23904, 2768, 29889, 450, 26935, 4933, 338, 7180, 373, 263, 7101, 411, 263, 4796, 3239, 29889, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [ ]:
# we fill the -100 indices with our pad token
tokenizer.decode(dataset[0]['input_ids'])

'<s> A modern coffee machine with a digital display and two white coffee cups filled with coffee is shown. The machine has a stainless steel finish and is accompanied by a milk frothing pitcher with a white liquid inside. The coffee machine is placed on a surface with a white background.</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s>'

In [ ]:
# custom collate
from concurrent.futures import ThreadPoolExecutor
import requests
import io
from PIL import Image

def custom_collate_fn(inputs):
  """
  collator for downloading images at runtime so we can work concurrently instead
  of downloading all images at once.
  """
  # list of dict to dict of lists
  inputs = {k: [i[k] for i in inputs] for k in inputs[0].keys()}

  # download imgages
  def download_img(url):
    try:
      img = Image.open(io.BytesIO(requests.get(url, timeout=3).content)).convert("RGB")
      if not len(img.size) == 2 or img.size[:2] <= (1, 1):
        return None
      else:
        return img
    except:
      return None

  urls = inputs['urls']
  with ThreadPoolExecutor(max_workers=8) as executor:
    imgs = list(executor.map(download_img, urls))

  # drop any inputs which we couldn't download images for
  bad_ids = [i for i, im in enumerate(imgs) if im is None]
  bad_ids = bad_ids[::-1]

  input_ids = inputs['input_ids']
  attention_mask = inputs['attention_mask']
  labels = inputs['labels']

  for i in bad_ids:
    input_ids.pop(i)
    attention_mask.pop(i)
    labels.pop(i)
    imgs.pop(i)

  return {
      'input_ids': torch.tensor(input_ids),
      'attention_mask': torch.tensor(attention_mask),
      'labels': torch.tensor(labels),
      'images': imgs
  }


In [ ]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]

train_dataset = CustomDataset(dataset)
dl = DataLoader(train_dataset, batch_size=16, collate_fn=custom_collate_fn)

# note: the length of the batch mayb be shorter than our batch size if some images didn't download!
next(iter(dl))['images']

[<PIL.Image.Image image mode=RGB size=180x230>,
 <PIL.Image.Image image mode=RGB size=400x40>,
 <PIL.Image.Image image mode=RGB size=256x256>,
 <PIL.Image.Image image mode=RGB size=390x390>,
 <PIL.Image.Image image mode=RGB size=170x170>,
 <PIL.Image.Image image mode=RGB size=600x900>,
 <PIL.Image.Image image mode=RGB size=1080x1080>,
 <PIL.Image.Image image mode=RGB size=236x330>,
 <PIL.Image.Image image mode=RGB size=250x250>,
 <PIL.Image.Image image mode=RGB size=810x400>,
 <PIL.Image.Image image mode=RGB size=733x413>,
 <PIL.Image.Image image mode=RGB size=270x152>,
 <PIL.Image.Image image mode=RGB size=320x180>,
 <PIL.Image.Image image mode=RGB size=1600x1000>]

In [ ]:
# get dataset ready for model training

In [ ]:
from transformers import Trainer, TrainingArguments
from torch.utils.data import DataLoader
import functools

class CustomTrainer(Trainer):
  """
  `Trainer` class automatically gets rid of columns in the self.train_dataset which aren't used
  by the model forward
  """
  def get_train_dataloader(self):
      return DataLoader(
          self.train_dataset,
          batch_size=self.args.train_batch_size,
          collate_fn=custom_collate_fn,
          shuffle=True,
          drop_last=self.args.dataloader_drop_last,
          num_workers=self.args.dataloader_num_workers,
      )

In [ ]:
training_args = TrainingArguments(
    output_dir="./tiny_vlm",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=5e-04,
    max_grad_norm=1.0,
    max_steps=1100,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr": 5e-05},
    warmup_ratio=0.03,
    logging_strategy="steps",
    logging_steps=25,
    seed=42,
    dataloader_num_workers=2,
    label_names=["labels"],
    report_to="none",
    dataloader_pin_memory=True,
    #fp16=True,
    #half_precision_backend="auto",
    #lora_config = LoraConfig(r=4, lora_alpha=32, target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'], bias = 'none'),
)

In [ ]:
# make sure to freeze all model parameters except those in our connector!

for p in model.parameters():
    p.requires_grad = False

for p in model.connector.parameters():
    p.requires_grad = True

params = sum((p.numel() for p in model.parameters()))
trainable = sum((p.numel() for p in model.parameters() if p.requires_grad))
print(
    f"VLM with: {params/1e9:.1f}B params | {100*trainable/params:.2f}% trainable\n"
)

trainer = CustomTrainer(
    model=model,
    tokenizer=model.tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=None,
)

max_steps is given, it will override any value given in num_train_epochs


VLM with: 1.6B params | 1.64% trainable



In [ ]:
trainer.train()

/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/local/lib/python3.10/dist-packages/PIL/Image.py:996: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Step,Training Loss
25,2.165100
50,1.686200
75,1.667900
100,1.692900
125,1.659200
150,1.644900
175,1.585500
200,1.636000
225,1.615400
250,1.613700


/usr/local/lib/python3.10/dist-packages/PIL/Image.py:996: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connection.py", line 464, in getresponse
    assert_header_parsing(httplib_response.msg)
  File "/usr/local/lib/python3.10/dist-packages/urllib3/util/response.py", line 88, in assert_header_parsing
    raise HeaderParsingError(defects=defects, unparsed_data=unparsed_data)
urllib3.exceptions.HeaderParsingError: [MissingHeaderBodySeparatorDefect()], unparsed data: 'X-Frame-Options : SAMEORIGIN\r\nX-Content-Type-Options : nosniff\r\nX-Xss-Protection : 1; mode=block\r\nDate: Sun, 15 Sep 2024 11:51:10 GMT\r\nContent-Length: 97284\r\n\r\n'


KeyboardInterrupt: 

In [ ]:
torch.cuda.empty_cache()

lets try the model after training on this image


<img src="https://www.birds.cornell.edu/home/wp-content/uploads/2023/09/334289821-Baltimore_Oriole-Matthew_Plante.jpg" alt="Baltimore Oriole" width="600">


In [ ]:
url = "https://media.istockphoto.com/id/537241730/photo/up-view-in-financial-district.jpg?s=612x612&w=0&k=20&c=7KQm2fkV--RBIJGfjA5Isik6ktrt-rVwt46Qtu5yGMw="
img = Image.open(io.BytesIO(requests.get(url).content))

template = tokenizer.bos_token + "<|user|>\n{prompt}</s>\n<|assistant|>\n"
prompt = "what can you see in this image?"
inputs = tokenizer(template.format(prompt=prompt), return_tensors = 'pt', add_special_tokens=False)
inputs['labels'] = inputs['input_ids'].clone()
inputs['images'] = [img,]

for k, v in inputs.items():
    if isinstance(v, torch.Tensor):
        inputs[k] = v.to(model.device)

generated = model.generate(**inputs, temperature=0.2, max_new_tokens=64, do_sample=True)
tokenizer.batch_decode(generated)

['The image shows a cityscape with a modern building in the foreground, surrounded by a grid of streets and buildings. The sky is clear, with a few clouds in the distance. The cityscape is dominated by a tall, slender building with a sleek, modern design. The building is surrounded']

In [ ]:
# sanity check
tokenizer.decode(inputs['input_ids'][0])

'<s> <|user|>\ndescribe this image briefly.</s> \n<|assistant|>\n'